- 这个脚本的目的是用来获取CAZy数据库中有功能注释酶的GT-A结构域的
- 选取三种方法结合的操作，与发表在Science上TED的文章处理方式一样，用来修正某一个具体工具可能存在的预测不准确问题
- 三个工具分别为：Merizo、Chainsaw、UniDoc
- 整体的流程分为：结构预测（pLDDT>60）->结构域划分->结构预测（pLDDT>70）
- 全部手动处理，按照elife文章做序列谱，只有序列谱中有保守基序的才是GT-A

# 确认家族的GT-A折叠类型

确认家族的GT-A的真实性，从DXD入手，但是在GT116上看到了DXK的序列，有点子离谱

In [127]:
# # 提取家族序列
# from Bio import SeqIO
# from tqdm import tqdm

# family_name = 'GT116'

# i = 0

# with open(f"./data/cazy_annot_domain/{family_name}.fa", "w") as out_handle:
#     for record in tqdm(SeqIO.parse("./data/cazy_annot_domain/CAZyDB.07142024.fa", "fasta"), total=3613704):
#         if f"|{family_name}\n" in record.id+'\n' or f"|{family_name}|" in record.id or f"|{family_name}_" in record.id:
#             if 'X' in record.seq or 'Z' in record.seq or 'B' in record.seq or 'J' in record.seq or 'O' in record.seq or 'U' in record.seq:
#                 continue
#             if len(record.seq) > 700:
#                 continue
#             SeqIO.write(record, out_handle, "fasta")
#             i += 1

# print(f"Total sequences extracted: {i}")


In [ ]:
# # 绘制家族的序列谱
# # conda activate plottree
# print(f"cdhit -c 0.7 -i {family_name}.fa -o {family_name}_sub.fa -T 190 -M 30000")
# print(f"mafft --thread 190 --op 10 --ep 0.2 --auto {family_name}.fa > {family_name}.aln")


# 提取带有活性注释的GT-A序列和结构

- 提取方法分为以下步骤：
- （0）在进行所有处理前，清晰是NCBI来源的，并去除非标准氨基酸和重复。
- （1）首先是获取domain，这里就要分为两个部分处理，一个是能够从elife的数据中找到domain的，另一个是找不到domain的
- （2）针对找不到domain的，单独一个文件，在文件末尾放上10个有domain的示例，用序列比对来划分domain
- （3）如果是单一功能的，就使用所有的序列；如果是多功能的，就按70%序列相似性划分功能
- （4）domain长度在140~350
- （5）进行结构预测，并筛选pLDDT>80的结构并手动检查结构合理性

In [ ]:
# # 提取家族序列，并清洗
# from Bio import SeqIO
# from tqdm import tqdm

# family_name = 'GT6'

# i = 0

# with open(f"./data/cazy_annot_domain/{family_name}.fa", "w") as out_handle:
#     for record in tqdm(SeqIO.parse("./data/cazy_annot_domain/CAZyDB.07142024.fa", "fasta"), total=3613704):
#         if f"|{family_name}\n" in record.id+'\n' or f"|{family_name}|" in record.id or f"|{family_name}_" in record.id:
#             if 'X' in record.seq or 'Z' in record.seq or 'B' in record.seq or 'J' in record.seq or 'O' in record.seq or 'U' in record.seq:
#                 continue
#             if '.' not in record.id.split('|')[0]:
#                 continue
#             SeqIO.write(record, out_handle, "fasta")
#             i += 1

# print(f"Total sequences extracted: {i}")

# print(f"cdhit -c 1 -i {family_name}.fa -o {family_name}_sub.fa -T 190 -M 30000")

100%|██████████| 3613704/3613704 [00:37<00:00, 97494.63it/s] 

Total sequences extracted: 899
cdhit -c 1 -i GT6.fa -o GT6_sub.fa -T 190 -M 30000


## One Function Type

In [ ]:
# # 从elife文章中获取domain序列
# from Bio import SeqIO
# import pandas as pd
# from Bio.SeqRecord import SeqRecord
# from Bio.Seq import Seq

# domain_count = 0
# full_count = 0

# df_elife = pd.read_csv('./data/cazy_annot_domain/GTA_nr.tsv', sep='\t')
# elige_dic = {}
# for i in range(len(df_elife)):
#     elige_dic[df_elife['SequenceID'][i]] = (df_elife['DomainSequence'][i], df_elife['FullSequence'][i])

# # 划分domain和full序列
# domain_handle =  open(f"./data/cazy_annot_domain/{family_name}_domain.fa", "w")
# full_handle =  open(f"./data/cazy_annot_domain/{family_name}_full.fa", "w")
# for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_sub.fa", "fasta")):
#     if record.id.split('|')[0] in elige_dic:
#         domain_seq, full_seq = elige_dic[record.id.split('|')[0]]
#         if record.seq != full_seq:
#             print(f"Error! {record.id} domain sequence not equal to full sequence!")
#             print(domain_seq)
#             print(full_seq)
#             continue
#         if 'X' in domain_seq or 'Z' in domain_seq or 'B' in domain_seq or 'J' in domain_seq or 'O' in domain_seq or 'U' in domain_seq:
#             print(f"Error! {record.id} domain sequence contains invalid characters!")
#             continue
#         record.seq = Seq(domain_seq)
#         record.id = record.id.replace('.','_')
#         SeqIO.write(record, domain_handle, "fasta")
#         domain_count += 1
#     else:
#         record.id = record.id.replace('.','_')
#         SeqIO.write(record, full_handle, "fasta")
#         full_count += 1

# domain_handle.close()

# # 写入10个示例domain序列
# i = 0
# for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_domain.fa", "fasta")):
#     if i >= 10:
#         break
#     record.id = 'example_' + str(i)
#     SeqIO.write(record, full_handle, "fasta")
#     i += 1

# full_handle.close()

# print(f"mafft --thread 190 --op 10 --ep 0.2 --auto {family_name}_full.fa > {family_name}.aln")
# print(f"Domain sequences: {domain_count}, full sequences: {full_count}")
# print("请人工检查比对结果，去除非domain的序列，检查.aln文件")

115it [00:00, 31833.75it/s]
10it [00:00, 29289.83it/s]

mafft --thread 190 --op 10 --ep 0.2 --auto GT12_full.fa > GT12.aln
Domain sequences: 18, full sequences: 97
请人工检查比对结果，去除非domain的序列，检查.aln文件


In [ ]:
# # 强力整合人工选取domain和电脑的
# import os

# out_handle = open(f"./data/cazy_annot_domain/{family_name}_d.fa", "w")
# excel_handle = open(f"./data/cazy_annot_domain/{family_name}_conclusion.tsv", "w")
# excel_handle.write("ID\tFamily\tLength\n")

# small_count = 0
# middle_count = 0
# large_count = 0

# for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_domain.fa", "fasta")):
#     if len(record.seq) > 140 and len(record.seq) < 350 and 'example_' not in record.id:
#         genbank_id = record.id.split('|')[0]
#         f_name = record.id.split(genbank_id+'|')[1]
#         excel_handle.write(f"{genbank_id}\t{f_name}\t{len(record.seq)}\n")
#         record.id = family_name + '_' + genbank_id
#         record.description = ""
#         SeqIO.write(record, out_handle, "fasta")
#         middle_count += 1
#     elif len(record.seq) < 140:
#         small_count += 1
#     elif len(record.seq) > 350:
#         large_count += 1

# for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}.aln", "fasta")):
#     record.seq = record.seq.replace('-','')
#     if len(record.seq) > 140 and len(record.seq) < 350 and 'example_' not in record.id:
#         record.id = record.id.split('/')[0]
#         genbank_id = record.id.split('|')[0]
#         f_name = record.id.split(genbank_id+'|')[1]
#         excel_handle.write(f"{genbank_id}\t{f_name}\t{len(record.seq)}\n")
#         record.id = family_name + '_' + genbank_id
#         record.description = ""
#         SeqIO.write(record, out_handle, "fasta")
#         middle_count += 1
#     elif len(record.seq) < 140:
#         small_count += 1
#     elif len(record.seq) > 350:
#         large_count += 1


# out_handle.close()
# excel_handle.close()

# os.makedirs(f"./data/cazy_annot_domain/{family_name}/", exist_ok=True)

# print(f"Small sequences: {small_count}, middle sequences: {middle_count}, large sequences: {large_count}")

# # export CUDA_VISIBLE_DEVICES=1

# print(f"cdhit -c 1 -i {family_name}_d.fa -o {family_name}_d_sub.fa -T 190 -M 30000")

# print(f"nohup python /home/admin123/esmfold/esm-main/scripts/fold.py \
#         -i {family_name}_d_sub.fa -o {family_name} -m /home/admin123/.cache/torch/hub/ \
#         --num-recycles 4 &")


18it [00:00, 31549.30it/s]
107it [00:00, 20154.96it/s]

Small sequences: 14, middle sequences: 101, large sequences: 0
cdhit -c 1 -i GT12_d.fa -o GT12_d_sub.fa -T 190 -M 30000
nohup python /home/admin123/esmfold/esm-main/scripts/fold.py         -i GT12_d_sub.fa -o GT12 -m /home/admin123/.cache/torch/hub/         --num-recycles 4 &


In [ ]:
# # 筛选结构，生成最终的文件
# import os

# os.makedirs(f"./data/cazy_annot_domain/{family_name}_r/", exist_ok=True)
# out_handle = open(f"./data/cazy_annot_domain/{family_name}_domain.fa", "w")
# excel_handle = open(f"./data/cazy_annot_domain/{family_name}_metadata.tsv", "w")
# excel_handle.write("ID\tFamily\tLength\tpLDDT\n")

# metadata_dict = {}
# df = pd.read_csv(f"./data/cazy_annot_domain/{family_name}_conclusion.tsv", sep='\t')
# for i in range(0,df.shape[0]):
#     metadata_dict[df['ID'][i]] = (df['Family'][i], df['Length'][i])
# sequence_dict = {}
# for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_d_sub.fa", "fasta")):
#     record.description = ""
#     sequence_dict[record.id] = record

# with open('./data/cazy_annot_domain/nohup.out', 'r')as f:
#     lines = f.readlines()

# for i,line in enumerate(lines):
#     if i <= 3:
#         continue
#     pLDDT = line.split(', pLDDT ')[1].split(', pTM ')[0]
#     if float(pLDDT) < 80:
#         continue
#     f_name = line.split('Predicted structure for ')[1].split(' with length ')[0]
#     f_name = f_name.split(f"{family_name}_")[1]
#     os.system(f"cp ./data/cazy_annot_domain/{family_name}/{family_name}_{f_name}.pdb ./data/cazy_annot_domain/{family_name}_r/")
#     temp1, temp2 = metadata_dict[f_name]
#     excel_handle.write(f"{f_name}\t{temp1}\t{temp2}\t{pLDDT}\n")
#     SeqIO.write(sequence_dict[f"{family_name}_{f_name}"], out_handle, "fasta")

# excel_handle.close()
# out_handle.close()


93it [00:00, 92368.05it/s]


In [ ]:
# # ==========脚本内容：Protein_align.py==========
# import subprocess
# import os
# from tqdm import tqdm

# def get_all_files(directory):
#     all_files = []
#     # 遍历目录及其子目录
#     for root, dirs, files in os.walk(directory):
#         # 将每个文件的绝对路径添加到列表中
#         for file in files:
#             file_path = os.path.join(root, file)
#             all_files.append(file_path)
#     return all_files

# def delete_files_in_folder(folder_path):
#     # 获取文件夹下所有文件和子文件夹
#     for root, dirs, files in os.walk(folder_path):
#         # 删除所有文件
#         for file in files:
#             file_path = os.path.join(root, file)
#             os.remove(file_path)

# def delete_special_files_in_folder(folder_path, prefix, fila_n):
#     # 获取文件夹下所有文件和子文件夹
#     for root, dirs, files in os.walk(folder_path):
#         # 删除所有文件
#         for file in files:
#             if file.endswith(prefix) and file.startswith(fila_n):
#                 file_path = os.path.join(root, file)
#                 os.remove(file_path)

# directory = f"./data/cazy_annot_domain/{family_name}_r/"
# storage_direcroey = f"./data/cazy_annot_domain/{family_name}_aln/"
# os.makedirs(storage_direcroey, exist_ok=True)
# all_files = get_all_files(directory)
# delete_files_in_folder(storage_direcroey)
# # USalign结构比对==========
# exe_path = './data/exe/USalign'
# cluster_center_pdb = './data/elife_cluster_center.pdb'
# # USalign AAA.pdb cluster.pdb -o AAA
# for file in tqdm(all_files):
#     file = file.split('/')[-1]
#     temp1 = os.path.join(directory, file)
#     temp2 = os.path.join(storage_direcroey, file.rstrip('.pdb'))
#     subprocess.run([exe_path, temp1, cluster_center_pdb, '-o', temp2], stdout=subprocess.DEVNULL)
#     delete_special_files_in_folder(storage_direcroey, '.pml', file.rstrip('.pdb'))

# print("Pleace download the file of '_aln', '_domain.fa', '_metadata.tsv''")

100%|██████████| 32/32 [00:04<00:00,  7.19it/s]

Pleace download the file of '_aln', '_domain.fa', '_metadata.tsv''


## Many Function Type

In [ ]:
# # 从elife文章中获取domain序列 ========== 多功能特调 ==========
# from Bio import SeqIO
# import pandas as pd
# from Bio.SeqRecord import SeqRecord
# from Bio.Seq import Seq

# domain_count = 0
# full_count = 0

# df_elife = pd.read_csv('./data/cazy_annot_domain/GTA_nr.tsv', sep='\t')
# elige_dic = {}
# for i in range(len(df_elife)):
#     elige_dic[df_elife['SequenceID'][i]] = (df_elife['DomainSequence'][i], df_elife['FullSequence'][i])

# # 划分domain和full序列
# domain_handle =  open(f"./data/cazy_annot_domain/{family_name}_domain.fa", "w")
# full_handle =  open(f"./data/cazy_annot_domain/{family_name}_full.fa", "w")
# for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_sub.fa", "fasta")):
#     if record.id.split('|')[0] in elige_dic:
#         domain_seq, full_seq = elige_dic[record.id.split('|')[0]]
#         if record.seq != full_seq:
#             print(f"Error! {record.id} domain sequence not equal to full sequence!")
#             print(domain_seq)
#             print(full_seq)
#             continue
#         if 'X' in domain_seq or 'Z' in domain_seq or 'B' in domain_seq or 'J' in domain_seq or 'O' in domain_seq or 'U' in domain_seq:
#             print(f"Error! {record.id} domain sequence contains invalid characters!")
#             continue
#         record.seq = Seq(domain_seq)
#         record.id = record.id.replace('.','_')
#         SeqIO.write(record, domain_handle, "fasta")
#         domain_count += 1
#     else:
#         record.id = record.id.replace('.','_')
#         SeqIO.write(record, full_handle, "fasta")
#         full_count += 1

# domain_handle.close()

# # ========== 多功能特调 ==========
# # 写入活性数据
# df = pd.read_excel('./data/cazy_annot_domain/表征数据.xlsx')
# df = df.loc[df['Family'] == family_name]
# df.reset_index(drop=True, inplace=True)
# Characterized_dic = {}
# for record in SeqIO.parse("./data/cazy_annot_domain/sequence.fasta", "fasta"):
#     Characterized_dic[record.id] = record.seq
# for i in range(0, df.shape[0]):
#     temp_id = df['GenBank'][i].replace('.','_')
#     temp_seq = Characterized_dic[temp_id]
#     record = SeqRecord(Seq(temp_seq), id='Characterized_' + temp_id, description="")
#     SeqIO.write(record, full_handle, "fasta")
# # ========== 多功能特调 ==========

# # 写入10个示例domain序列
# i = 0
# for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_domain.fa", "fasta")):
#     if i >= 10:
#         break
#     record.id = 'example_' + str(i)
#     SeqIO.write(record, full_handle, "fasta")
#     i += 1

# full_handle.close()

# print(f"mafft --thread 190 --op 10 --ep 0.2 --auto {family_name}_full.fa > {family_name}.aln")
# print(f"Domain sequences: {domain_count}, full sequences: {full_count}")
# print("请人工检查比对结果，去除非domain的序列，检查.aln文件")

530it [00:00, 29381.19it/s]
10it [00:00, 30240.12it/s]

mafft --thread 190 --op 10 --ep 0.2 --auto GT6_full.fa > GT6.aln
Domain sequences: 285, full sequences: 245
请人工检查比对结果，去除非domain的序列，检查.aln文件


In [ ]:
# # # 从elife文章中获取domain序列 ========== 多功能特调 ========== GT2这种上万条的特别版，将序列以1w为单位进行拆分 ==========
# # from Bio import SeqIO
# # import pandas as pd
# # from Bio.SeqRecord import SeqRecord
# # from Bio.Seq import Seq
# # from tqdm import tqdm


# # sub_dic = {}
# # for record in SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_sub.fa", "fasta"):
# #     sub_dic[record.id] = record.seq
# # chunk_size = 10000
# # items = list(sub_dic.items())
# # for i in range(0, len(items), chunk_size):
# #         chunk = items[i:i+chunk_size]
# #         file_num = i // chunk_size + 1
# #         # small_full_handle =  open(f"./data/cazy_annot_domain/{family_name}_{file_num}_sub.fa", "w") # 运行一次
# #         # for key, value in chunk:
# #         #     record = SeqRecord(Seq(value), id=key, description="")
# #         #     SeqIO.write(record, small_full_handle, "fasta")
# #         # small_full_handle.close()

# # print(f"File is split to {file_num} files, each containing {chunk_size} sequences.")
# # sub_num = 20 # =================================================================================================================== 关键

# # df_elife = pd.read_csv('./data/cazy_annot_domain/GTA_nr.tsv', sep='\t')
# # elige_dic = {}
# # for i in range(len(df_elife)):
# #     elige_dic[df_elife['SequenceID'][i]] = (df_elife['DomainSequence'][i], df_elife['FullSequence'][i])
# # domain_count = 0
# # full_count = 0
# # # 划分domain和full序列
# # domain_handle =  open(f"./data/cazy_annot_domain/{family_name}_domain.fa", "w")
# # full_handle =  open(f"./data/cazy_annot_domain/{family_name}_full.fa", "w")
# # for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_{sub_num}_sub.fa", "fasta")):
# #     if record.id.split('|')[0] in elige_dic:
# #         domain_seq, full_seq = elige_dic[record.id.split('|')[0]]
# #         if record.seq != full_seq:
# #             # print(f"Error! {record.id} domain sequence not equal to full sequence!")
# #             # print(domain_seq)
# #             # print(full_seq)
# #             continue
# #         if 'X' in domain_seq or 'Z' in domain_seq or 'B' in domain_seq or 'J' in domain_seq or 'O' in domain_seq or 'U' in domain_seq:
# #             # print(f"Error! {record.id} domain sequence contains invalid characters!")
# #             continue
# #         record.seq = Seq(domain_seq)
# #         record.id = record.id.replace('.','_')
# #         SeqIO.write(record, domain_handle, "fasta")
# #         domain_count += 1
# #     else:
# #         record.id = record.id.replace('.','_')
# #         SeqIO.write(record, full_handle, "fasta")
# #         full_count += 1

# # domain_handle.close()

# # # ========== 多功能特调 ==========
# # # 写入活性数据
# # df = pd.read_excel('./data/cazy_annot_domain/表征数据.xlsx')
# # df = df.loc[df['Family'] == family_name]
# # df.reset_index(drop=True, inplace=True)
# # Characterized_dic = {}
# # for record in SeqIO.parse("./data/cazy_annot_domain/sequence.fasta", "fasta"):
# #     Characterized_dic[record.id] = record.seq
# # for i in range(0, df.shape[0]):
# #     temp_id = df['GenBank'][i].replace('.','_')
# #     temp_seq = Characterized_dic[temp_id]
# #     record = SeqRecord(Seq(temp_seq), id='Characterized_' + temp_id, description="")
# #     SeqIO.write(record, full_handle, "fasta")
# # # ========== 多功能特调 ==========

# # # 写入10个示例domain序列
# # i = 0
# # for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_domain.fa", "fasta")):
# #     if i >= 50: # ================================================================================================= 关键，调整个数，减少aln中因为长度被筛选的序列
# #         break
# #     record.id = 'example_' + str(i)
# #     SeqIO.write(record, full_handle, "fasta")
# #     i += 1

# # full_handle.close()

# # print(f"mafft --thread 190 --op 10 --ep 0.2 --auto {family_name}_full.fa > {family_name}.aln")
# # print(f"Domain sequences: {domain_count}, full sequences: {full_count}")
# # print("请人工检查比对结果，去除非domain的序列，检查.aln文件")
# # print("在正式运行的时候关掉了error输出")

# # # 已经完成 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20
# # # 正在进行 
# # # mafft --thread 190 --op 10 --ep 0.2 --auto GT2_1_full.fa > GT2.aln

File is split to 20 files, each containing 10000 sequences.


9793it [00:00, 35507.34it/s]
50it [00:00, 18493.40it/s]

mafft --thread 190 --op 10 --ep 0.2 --auto GT2_full.fa > GT2.aln
Domain sequences: 705, full sequences: 9088
请人工检查比对结果，去除非domain的序列，检查.aln文件
在正式运行的时候关掉了error输出


In [ ]:
# # 强力整合人工选取domain和电脑的
# import os
# from tqdm import tqdm
# from Bio import SeqIO
# import pandas as pd
# from Bio.SeqRecord import SeqRecord
# from Bio.Seq import Seq

# out_handle = open(f"./data/cazy_annot_domain/{family_name}_d.fa", "w")
# Characterized_handle = open(f"./data/cazy_annot_domain/{family_name}_Characterized.fa", "w")
# excel_handle = open(f"./data/cazy_annot_domain/{family_name}_conclusion.tsv", "w")
# excel_handle.write("ID\tFamily\tLength\n")
# Characterized_excel_handle = open(f"./data/cazy_annot_domain/{family_name}_Characterized_conclusion.tsv", "w")
# Characterized_excel_handle.write("ID\tFamily\tLength\n")

# small_count = 0
# middle_count = 0
# large_count = 0

# for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_domain.fa", "fasta")):
#     if len(record.seq) > 140 and len(record.seq) < 350 and 'example_' not in record.id:
#         genbank_id = record.id.split('|')[0]
#         f_name = record.id.split(genbank_id+'|')[1]
#         excel_handle.write(f"{genbank_id}\t{f_name}\t{len(record.seq)}\n")
#         record.id = family_name + '_' + genbank_id
#         record.description = ""
#         SeqIO.write(record, out_handle, "fasta")
#         middle_count += 1
#     elif len(record.seq) < 140:
#         small_count += 1
#     elif len(record.seq) > 350:
#         large_count += 1

# for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}.aln", "fasta")):
#     record.seq = record.seq.replace('-','')
#     # ========== 多功能特调 ==========
#     if len(record.seq) > 140 and len(record.seq) < 350 and 'Characterized_' in record.id:
#         record.id = record.id.split('/')[0]
#         genbank_id = record.id.split('Characterized_')[1]
#         f_name = family_name
#         Characterized_excel_handle.write(f"{genbank_id}\t{f_name}\t{len(record.seq)}\n")
#         record.id = family_name + '_' + genbank_id
#         record.description = ""
#         SeqIO.write(record, Characterized_handle, "fasta")
#         middle_count += 1
#     # ========== 多功能特调 ==========
#     elif len(record.seq) > 140 and len(record.seq) < 350 and 'example_' not in record.id:
#         record.id = record.id.split('/')[0]
#         genbank_id = record.id.split('|')[0]
#         f_name = record.id.split(genbank_id+'|')[1]
#         excel_handle.write(f"{genbank_id}\t{f_name}\t{len(record.seq)}\n")
#         record.id = family_name + '_' + genbank_id
#         record.description = ""
#         SeqIO.write(record, out_handle, "fasta")
#         middle_count += 1
#     elif len(record.seq) < 140:
#         small_count += 1
#     elif len(record.seq) > 350:
#         large_count += 1


# out_handle.close()
# excel_handle.close()
# Characterized_handle.close()
# Characterized_excel_handle.close()

# print(f"Small sequences: {small_count}, middle sequences: {middle_count}, large sequences: {large_count}")
# print(f"cdhit -c 1 -i {family_name}_d.fa -o {family_name}_d_sub.fa -T 190 -M 30000")
# print(f"cdhit -c 1 -i {family_name}_Characterized.fa -o {family_name}_Characterized_sub.fa -T 190 -M 30000")


285it [00:00, 27090.07it/s]
740it [00:00, 24401.02it/s]

Small sequences: 49, middle sequences: 962, large sequences: 4
cdhit -c 1 -i GT6_d.fa -o GT6_d_sub.fa -T 190 -M 30000
cdhit -c 1 -i GT6_Characterized.fa -o GT6_Characterized_sub.fa -T 190 -M 30000


In [ ]:
# # 多功能家族最关键的70%序列相似性部分 Step1
# # ========== 多功能特调 ==========
# wait_aln_handle = open(f"./data/cazy_annot_domain/{family_name}_wait_aln.fa", "w")
# for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_d_sub.fa", "fasta")):
#     record.description = ""
#     SeqIO.write(record, wait_aln_handle, "fasta")
# for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_Characterized_sub.fa", "fasta")):
#     record.description = ""
#     record.id = "Characterized_" + record.id
#     SeqIO.write(record, wait_aln_handle, "fasta")
# wait_aln_handle.close()

# print(f"mafft --thread 190 --op 10 --ep 0.2 --auto {family_name}_wait_aln.fa > {family_name}_wait_aln.aln")
# # ========== 多功能特调 ==========

446it [00:00, 34706.76it/s]
225it [00:00, 31021.94it/s]

mafft --thread 190 --op 10 --ep 0.2 --auto GT6_wait_aln.fa > GT6_wait_aln.aln


In [ ]:
# # 多功能家族最关键的70%序列相似性部分 Step1
# from itertools import combinations
# from multiprocessing import Pool, cpu_count

# def calculate_similarity(pair):
#     (key1, value1), (key2, value2) = pair
#     total_site = 0
#     similarity_site = 0
#     # 对齐两条序列并计算相似度
#     for site1, site2 in zip(value1, value2):
#         if site1 == '-' and site2 == '-':
#             continue
#         else:
#             total_site += 1
#             if site1 == site2:
#                 similarity_site += 1
#     similarity = similarity_site / total_site * 100
#     return f"{key1}\t{key2}\t{similarity:.2f}\n"
# # ========== 多功能特调 ==========
# out_handle = open(f"./data/cazy_annot_domain/{family_name}_70_similarity.tsv", "w", encoding='gbk')
# out_handle.write("GenBank\t活性\tNDP-Sugar活性\tProteinName\tCharacterized_GenBank\n")
# predict_handle = open(f"./data/cazy_annot_domain/{family_name}_wait_predict.fa", "w")

# Characterized_matadata = {}
# df = pd.read_excel('./data/cazy_annot_domain/表征数据.xlsx')
# df = df.loc[df['Family'] == family_name]
# df.reset_index(drop=True, inplace=True)
# for i in range(0, df.shape[0]):
#     temp_id = df['GenBank'][i].replace('.','_')
#     Characterized_matadata[temp_id] = (df['活性'][i], df['NDP-Sugar活性'][i], df['ProteinName'][i])

# cazy_dic = {}
# Characterized_dic = {}
# for record in SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_wait_aln.aln", "fasta"):
#     record.description = ""
#     if 'Characterized_' in record.id:
#         Characterized_dic[record.id] = str(record.seq)
#     else:
#         cazy_dic[record.id] = str(record.seq)


# temp_handle = open(f"./data/cazy_annot_domain/{family_name}_temp.tsv", "w")
# temp_handle.write("Key1\tKey2\tSimilarity\n")
# pairs = []
# for key1, value1 in tqdm(cazy_dic.items()):
#     for key2, value2 in Characterized_dic.items():
#         pairs.append(((key1, value1), (key2, value2)))
# # 使用多进程并行计算相似度
# with Pool(processes=cpu_count()) as pool:
#     for result in tqdm(pool.imap(calculate_similarity, pairs), total=len(pairs)):
#         temp_handle.write(result)
    
# predict_count = 0
# df = pd.read_csv(f"./data/cazy_annot_domain/{family_name}_temp.tsv", sep='\t')
# for group_name, group_df in tqdm(df.groupby('Key1')):
#     group_df.sort_values(by='Similarity', ascending=False, inplace=True)
#     group_df.reset_index(drop=True, inplace=True)
#     if group_df['Similarity'][0] < 70:
#         continue
#     temp_Characterized_genbank = group_df['Key2'][0].split('Characterized_')[1].split(f"{family_name}_")[1]
#     matadata_1, matadata_2, matadata_3 = Characterized_matadata[temp_Characterized_genbank]
#     out_handle.write(f"{group_name}\t{matadata_1}\t{matadata_2}\t{matadata_3}\t{temp_Characterized_genbank}\n")
#     record = SeqRecord(Seq(cazy_dic[group_name].replace('-','')), id=group_name, description="")
#     SeqIO.write(record, predict_handle, "fasta")
#     predict_count += 1

# temp_handle.close()
# out_handle.close()
# predict_handle.close()
# # ========== 多功能特调 ==========

# # 结构预测
# os.makedirs(f"./data/cazy_annot_domain/{family_name}/", exist_ok=True)
# print(f"Predict structure for {predict_count}.")
# # export CUDA_VISIBLE_DEVICES=1
# print(f"nohup python /home/admin123/esmfold/esm-main/scripts/fold.py \
#         -i {family_name}_wait_predict.fa -o {family_name} -m /home/admin123/.cache/torch/hub/ \
#         --num-recycles 4 &")

100%|██████████| 446/446 [00:00<00:00, 2546.23it/s]

Predict structure for 321.
nohup python /home/admin123/esmfold/esm-main/scripts/fold.py         -i GT6_wait_predict.fa -o GT6 -m /home/admin123/.cache/torch/hub/         --num-recycles 4 &


In [ ]:
# # # GT2 这种活性特别多的结构的特调 =========================================================================================
# # print(f"cdhit -c 0.4 -n 2 -i {family_name}_wait_predict.fa -o {family_name}_wait_predict_sub.fa -T 190 -M 30000")
# # print(f"mafft --thread 190 --reorder --op 10 --ep 0.2 --auto {family_name}_wait_predict.fa > {family_name}_wait_predict.aln")
# # print("测试预测，方便调整参数")
# # os.makedirs(f"./data/cazy_annot_domain/{family_name}_test/", exist_ok=True)
# # print(f"python /home/admin123/esmfold/esm-main/scripts/fold.py \
# #         -i {family_name}_wait_predict_sub.fa -o {family_name}_test -m /home/admin123/.cache/torch/hub/ \
# #         --num-recycles 4")

cdhit -c 0.4 -n 2 -i GT2_wait_predict.fa -o GT2_wait_predict_sub.fa -T 190 -M 30000
mafft --thread 190 --reorder --op 10 --ep 0.2 --auto GT2_wait_predict.fa > GT2_wait_predict.aln
测试预测，方便调整参数
python /home/admin123/esmfold/esm-main/scripts/fold.py         -i GT2_wait_predict_sub.fa -o GT2_test -m /home/admin123/.cache/torch/hub/         --num-recycles 4


In [ ]:
# # # GT2 这种活性特别多的结构的特调 =========================================================================================
# # import subprocess
# # import os
# # from tqdm import tqdm

# # def get_all_files(directory):
# #     all_files = []
# #     # 遍历目录及其子目录
# #     for root, dirs, files in os.walk(directory):
# #         # 将每个文件的绝对路径添加到列表中
# #         for file in files:
# #             file_path = os.path.join(root, file)
# #             all_files.append(file_path)
# #     return all_files

# # def delete_files_in_folder(folder_path):
# #     # 获取文件夹下所有文件和子文件夹
# #     for root, dirs, files in os.walk(folder_path):
# #         # 删除所有文件
# #         for file in files:
# #             file_path = os.path.join(root, file)
# #             os.remove(file_path)

# # def delete_special_files_in_folder(folder_path, prefix, fila_n):
# #     # 获取文件夹下所有文件和子文件夹
# #     for root, dirs, files in os.walk(folder_path):
# #         # 删除所有文件
# #         for file in files:
# #             if file.endswith(prefix) and file.startswith(fila_n):
# #                 file_path = os.path.join(root, file)
# #                 os.remove(file_path)

# # directory = f"./data/cazy_annot_domain/{family_name}_test/"
# # storage_direcroey = f"./data/cazy_annot_domain/{family_name}_test_aln/"
# # os.makedirs(storage_direcroey, exist_ok=True)
# # all_files = get_all_files(directory)
# # delete_files_in_folder(storage_direcroey)
# # # USalign结构比对==========
# # exe_path = './data/exe/USalign'
# # cluster_center_pdb = './data/elife_cluster_center.pdb'
# # # USalign AAA.pdb cluster.pdb -o AAA
# # for file in tqdm(all_files):
# #     file = file.split('/')[-1]
# #     temp1 = os.path.join(directory, file)
# #     temp2 = os.path.join(storage_direcroey, file.rstrip('.pdb'))
# #     subprocess.run([exe_path, temp1, cluster_center_pdb, '-o', temp2], stdout=subprocess.DEVNULL)
# #     delete_special_files_in_folder(storage_direcroey, '.pml', file.rstrip('.pdb'))

# # print('Please download the file of "_test_aln" and "_wait_predict.aln"')

100%|██████████| 52/52 [00:09<00:00,  5.58it/s]

Please download the file of "_test_aln" and "_wait_predict.aln"


In [ ]:
# # # GT2 这种活性特别多的结构的特调 =========================================================================================
# # # 针对信息进行修正
# # predict_count = 0
# # conclusion_dic = {}
# # df = pd.read_csv(f"./data/cazy_annot_domain/{family_name}_conclusion.tsv", sep='\t')
# # for i in range(0,df.shape[0]):
# #     conclusion_dic[df['ID'][i]] = i

# # predict_handle = open(f"./data/cazy_annot_domain/{family_name}_wait_predict.fa", "w")
# # for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_wait_predict.aln", "fasta")):
# #     record.id = record.id.split('/')[0]
# #     record.seq = record.seq.replace('-','')
# #     record.description = ""
# #     SeqIO.write(record, predict_handle, "fasta")
# #     refine_id = record.id.split(f"{family_name}_")[1]
# #     refine_length = len(record.seq)
# #     df['Length'][conclusion_dic[refine_id]] = refine_length
# #     predict_count += 1

# # df.to_csv(f"./data/cazy_annot_domain/{family_name}_conclusion.tsv", sep='\t', index=False)
# # predict_handle.close()

# # # 结构预测
# # os.makedirs(f"./data/cazy_annot_domain/{family_name}/", exist_ok=True)
# # print(f"Predict structure for {predict_count}.")
# # # export CUDA_VISIBLE_DEVICES=1
# # print(f"nohup python /home/admin123/esmfold/esm-main/scripts/fold.py \
# #         -i {family_name}_wait_predict.fa -o {family_name} -m /home/admin123/.cache/torch/hub/ \
# #         --num-recycles 4 &")

0it [00:00, ?it/s]/home/admin123/software/anaconda3/envs/starG/lib/python3.6/site-packages/ipykernel_launcher.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
461it [00:00, 9216.98it/s]

Predict structure for 461.
nohup python /home/admin123/esmfold/esm-main/scripts/fold.py         -i GT2_wait_predict.fa -o GT2 -m /home/admin123/.cache/torch/hub/         --num-recycles 4 &


In [ ]:
# # 筛选结构，生成最终的文件
# import os

# os.makedirs(f"./data/cazy_annot_domain/{family_name}_r/", exist_ok=True)
# out_handle = open(f"./data/cazy_annot_domain/{family_name}_domain.fa", "w")
# excel_handle = open(f"./data/cazy_annot_domain/{family_name}_metadata.tsv", "w", encoding='gbk')
# excel_handle.write("活性\tNDP-Sugar活性\tGenBank\tAnootFamily\tLength\tpLDDT\tActivities\t模板\n")

# metadata_dict = {}
# df = pd.read_csv(f"./data/cazy_annot_domain/{family_name}_conclusion.tsv", sep='\t')
# for i in range(0,df.shape[0]):
#     metadata_dict[df['ID'][i]] = (df['Family'][i], df['Length'][i])
# metadata_dict_2 = {}
# df = pd.read_csv(f"./data/cazy_annot_domain/{family_name}_70_similarity.tsv", sep='\t', encoding='gbk')
# for i in range(0,df.shape[0]):
#     metadata_dict_2[df['GenBank'][i]] = (df['活性'][i], df['NDP-Sugar活性'][i], df['ProteinName'][i], df['Characterized_GenBank'][i])
# sequence_dict = {}
# for record in tqdm(SeqIO.parse(f"./data/cazy_annot_domain/{family_name}_wait_predict.fa", "fasta")):
#     record.description = ""
#     sequence_dict[record.id] = record

# with open('./data/cazy_annot_domain/nohup.out', 'r')as f:
#     lines = f.readlines()

# for i,line in enumerate(lines):
#     if i <= 3:
#         continue
#     pLDDT = line.split(', pLDDT ')[1].split(', pTM ')[0]
#     if float(pLDDT) < 80:
#         continue
#     f_name = line.split('Predicted structure for ')[1].split(' with length ')[0]
#     f_name = f_name.split(f"{family_name}_")[1]
#     os.system(f"cp ./data/cazy_annot_domain/{family_name}/{family_name}_{f_name}.pdb ./data/cazy_annot_domain/{family_name}_r/")
#     temp1, temp2 = metadata_dict[f_name]
#     temp3, temp4, temp5, temp6 = metadata_dict_2[f"{family_name}_{f_name}"]
#     excel_handle.write(f"{temp3}\t{temp4}\t{f_name}\t{temp1}\t{temp2}\t{pLDDT}\t{temp5}\t{temp6}\n")
#     SeqIO.write(sequence_dict[f"{family_name}_{f_name}"], out_handle, "fasta")

# excel_handle.close()
# out_handle.close()


321it [00:00, 12204.57it/s]


In [ ]:
# # ==========脚本内容：Protein_align.py==========
# import subprocess
# import os
# from tqdm import tqdm

# def get_all_files(directory):
#     all_files = []
#     # 遍历目录及其子目录
#     for root, dirs, files in os.walk(directory):
#         # 将每个文件的绝对路径添加到列表中
#         for file in files:
#             file_path = os.path.join(root, file)
#             all_files.append(file_path)
#     return all_files

# def delete_files_in_folder(folder_path):
#     # 获取文件夹下所有文件和子文件夹
#     for root, dirs, files in os.walk(folder_path):
#         # 删除所有文件
#         for file in files:
#             file_path = os.path.join(root, file)
#             os.remove(file_path)

# def delete_special_files_in_folder(folder_path, prefix, fila_n):
#     # 获取文件夹下所有文件和子文件夹
#     for root, dirs, files in os.walk(folder_path):
#         # 删除所有文件
#         for file in files:
#             if file.endswith(prefix) and file.startswith(fila_n):
#                 file_path = os.path.join(root, file)
#                 os.remove(file_path)

# directory = f"./data/cazy_annot_domain/{family_name}_r/"
# storage_direcroey = f"./data/cazy_annot_domain/{family_name}_aln/"
# os.makedirs(storage_direcroey, exist_ok=True)
# all_files = get_all_files(directory)
# delete_files_in_folder(storage_direcroey)
# # USalign结构比对==========
# exe_path = './data/exe/USalign'
# cluster_center_pdb = './data/elife_cluster_center.pdb'
# # USalign AAA.pdb cluster.pdb -o AAA
# for file in tqdm(all_files):
#     file = file.split('/')[-1]
#     temp1 = os.path.join(directory, file)
#     temp2 = os.path.join(storage_direcroey, file.rstrip('.pdb'))
#     subprocess.run([exe_path, temp1, cluster_center_pdb, '-o', temp2], stdout=subprocess.DEVNULL)
#     delete_special_files_in_folder(storage_direcroey, '.pml', file.rstrip('.pdb'))

# print("Pleace download the file of '_aln', '_domain.fa', '_metadata.tsv''")

100%|██████████| 294/294 [00:45<00:00,  6.42it/s]

Pleace download the file of '_aln', '_domain.fa', '_metadata.tsv''


# =====以下废物=====

In [ ]:
# # ==========脚本内容：Protein_align.py==========
# import subprocess
# import os
# from tqdm import tqdm

# def get_all_files(directory):
#     all_files = []
#     # 遍历目录及其子目录
#     for root, dirs, files in os.walk(directory):
#         # 将每个文件的绝对路径添加到列表中
#         for file in files:
#             file_path = os.path.join(root, file)
#             all_files.append(file_path)
#     return all_files

# def delete_files_in_folder(folder_path):
#     # 获取文件夹下所有文件和子文件夹
#     for root, dirs, files in os.walk(folder_path):
#         # 删除所有文件
#         for file in files:
#             file_path = os.path.join(root, file)
#             os.remove(file_path)

# def delete_special_files_in_folder(folder_path, prefix, fila_n):
#     # 获取文件夹下所有文件和子文件夹
#     for root, dirs, files in os.walk(folder_path):
#         # 删除所有文件
#         for file in files:
#             if file.endswith(prefix) and file.startswith(fila_n):
#                 file_path = os.path.join(root, file)
#                 os.remove(file_path)

# directory = f"/home/admin123/temp/man_structure_1/ttt/"
# storage_direcroey = f"/home/admin123/temp/man_structure_1/ttt_aln/"
# os.makedirs(storage_direcroey, exist_ok=True)
# all_files = get_all_files(directory)
# delete_files_in_folder(storage_direcroey)
# # USalign结构比对==========
# exe_path = './data/exe/USalign'
# cluster_center_pdb = './data/elife_cluster_center.pdb'
# # USalign AAA.pdb cluster.pdb -o AAA
# for file in tqdm(all_files):
#     file = file.split('/')[-1]
#     temp1 = os.path.join(directory, file)
#     temp2 = os.path.join(storage_direcroey, file.rstrip('.pdb'))
#     subprocess.run([exe_path, temp1, cluster_center_pdb, '-o', temp2], stdout=subprocess.DEVNULL)
#     delete_special_files_in_folder(storage_direcroey, '.pml', file.rstrip('.pdb'))

# print("Pleace download the file of '_aln', '_domain.fa', '_metadata.tsv''")

100%|██████████| 230/230 [00:18<00:00, 12.27it/s]

Pleace download the file of '_aln', '_domain.fa', '_metadata.tsv''
